# Fine-Tuning with LoRA & QLoRA Notebook

> Hands-on Build It and Exercises.

## Build It

We implement LoRA from scratch in pure PyTorch. No libraries. No magic. You'll build the LoRA layer, inject it into a model, train it, and merge the weights back.

### Step 1: The LoRA Layer

In [ ]:
```python

import torch

import torch.nn as nn

import math

class LoRALayer(nn.Module):

    def __init__(self, in_features, out_features, rank=8, alpha=16):

        super().__init__()

        self.rank = rank

        self.alpha = alpha

        self.scaling = alpha / rank

        self.A = nn.Parameter(torch.randn(in_features, rank) * (1 / math.sqrt(rank)))

        self.B = nn.Parameter(torch.zeros(rank, out_features))

    def forward(self, x):

        return (x @ self.A @ self.B) * self.scaling

In [ ]:
```

A is initialized with scaled random values. B is initialized to zero. The product BA starts at zero, so the model begins with its original behavior.

### Step 2: LoRA-Wrapped Linear Layer

In [ ]:
```python

class LinearWithLoRA(nn.Module):

    def __init__(self, linear, rank=8, alpha=16):

        super().__init__()

        self.linear = linear

        self.lora = LoRALayer(

            linear.in_features, linear.out_features, rank, alpha

        )

        for param in self.linear.parameters():

            param.requires_grad = False

    def forward(self, x):

        return self.linear(x) + self.lora(x)

In [ ]:
```

The original linear layer is frozen. Only the LoRA parameters (A and B) are trainable.

### Step 3: Inject LoRA into a Model

In [ ]:
```python

def inject_lora(model, target_modules, rank=8, alpha=16):

    for param in model.parameters():

        param.requires_grad = False

    lora_layers = {}

    for name, module in model.named_modules():

        if isinstance(module, nn.Linear):

            if any(t in name for t in target_modules):

                parent_name = ".".join(name.split(".")[:-1])

                child_name = name.split(".")[-1]

                parent = dict(model.named_modules())[parent_name]

                lora_linear = LinearWithLoRA(module, rank, alpha)

                setattr(parent, child_name, lora_linear)

                lora_layers[name] = lora_linear

    return lora_layers

In [ ]:
```

First, freeze every parameter in the model. Then walk the model tree, find linear layers matching your target names, and replace them with LoRA-wrapped versions. The LoRA A and B matrices are the only trainable parameters in the entire model.

### Step 4: Count Parameters

In [ ]:
```python

def count_parameters(model):

    total = sum(p.numel() for p in model.parameters())

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    frozen = total - trainable

    return {

        "total": total,

        "trainable": trainable,

        "frozen": frozen,

        "trainable_pct": 100 * trainable / total if total > 0 else 0

    }

In [ ]:
```

### Step 5: Merge Weights Back

In [ ]:
```python

def merge_lora_weights(model):

    for name, module in model.named_modules():

        if isinstance(module, LinearWithLoRA):

            with torch.no_grad():

                merged = (

                    module.lora.A @ module.lora.B

                ) * module.lora.scaling

                module.linear.weight.data += merged.T

            parent_name = ".".join(name.split(".")[:-1])

            child_name = name.split(".")[-1]

            if parent_name:

                parent = dict(model.named_modules())[parent_name]

            else:

                parent = model

            setattr(parent, child_name, module.linear)

In [ ]:
```

After merging, the LoRA layers are gone. The model is the same size as the original with the adaptation baked into the weights. No inference overhead.

### Step 6: Simulated QLoRA Quantization

In [ ]:
```python

def quantize_to_nf4(tensor, block_size=64):

    blocks = tensor.reshape(-1, block_size)

    scales = blocks.abs().max(dim=1, keepdim=True).values / 7.0

    scales = torch.clamp(scales, min=1e-8)

    quantized = torch.round(blocks / scales).clamp(-8, 7).to(torch.int8)

    return quantized, scales

def dequantize_from_nf4(quantized, scales, original_shape):

    dequantized = quantized.float() * scales

    return dequantized.reshape(original_shape)

In [ ]:
```

This simulates 4-bit quantization by mapping weights into 16 discrete levels within blocks of 64. Production QLoRA uses the bitsandbytes library for true NF4 on GPU.

### Step 7: Training Loop

In [ ]:
```python

def train_lora(model, data, epochs=5, lr=1e-3, batch_size=4):

    optimizer = torch.optim.AdamW(

        [p for p in model.parameters() if p.requires_grad], lr=lr

    )

    criterion = nn.MSELoss()

    losses = []

    for epoch in range(epochs):

        epoch_loss = 0.0

        n_batches = 0

        indices = torch.randperm(len(data["inputs"]))

        for i in range(0, len(indices), batch_size):

            batch_idx = indices[i:i + batch_size]

            x = data["inputs"][batch_idx]

            y = data["targets"][batch_idx]

            output = model(x)

            loss = criterion(output, y)

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            epoch_loss += loss.item()

            n_batches += 1

        avg_loss = epoch_loss / n_batches

        losses.append(avg_loss)

    return losses

In [ ]:
```

### Step 8: Full Demo

In [ ]:
```python

def demo():

    torch.manual_seed(42)

    d_model = 256

    n_classes = 10

    model = nn.Sequential(

        nn.Linear(d_model, 512),

        nn.ReLU(),

        nn.Linear(512, 512),

        nn.ReLU(),

        nn.Linear(512, n_classes),

    )

    n_samples = 500

    x = torch.randn(n_samples, d_model)

    y = torch.randint(0, n_classes, (n_samples,))

    y_onehot = torch.zeros(n_samples, n_classes).scatter_(1, y.unsqueeze(1), 1.0)

    data = {"inputs": x, "targets": y_onehot}

    params_before = count_parameters(model)

    lora_layers = inject_lora(

        model, target_modules=["0", "2"], rank=8, alpha=16

    )

    params_after = count_parameters(model)

    losses = train_lora(model, data, epochs=20, lr=1e-3)

    merge_lora_weights(model)

    params_merged = count_parameters(model)

    return {

        "params_before": params_before,

        "params_after": params_after,

        "params_merged": params_merged,

        "losses": losses,

    }

In [ ]:
```

The demo creates a small model, injects LoRA into two layers, trains it, and merges the weights back. The parameter count drops from full trainable to ~1% trainable during LoRA training, then returns to the original architecture after merging.

## Exercises

In [ ]:
1. **Rank ablation study.** Run the demo with ranks 2, 4, 8, 16, 32, and 64. Plot final loss vs. rank. Find the point of diminishing returns where doubling the rank no longer halves the loss. For a simple classification task on 256-dim features, this should be around r=8-16.

2. **Target module comparison.** Modify inject_lora to target only layer "0", only layer "2", only layer "4", and all three. Train each variant for 20 epochs. Compare convergence speed and final loss. This mirrors the real decision of targeting q_proj vs v_proj vs all linear layers.

3. **Quantization error analysis.** Take the trained model's weight matrices before and after quantize_to_nf4 / dequantize_from_nf4. Compute the mean squared error, max absolute error, and the correlation between original and reconstructed weights. Experiment with block_size values of 32, 64, 128, and 256.

4. **Multi-adapter serving.** Train two LoRA adapters on different subsets of the data (even indices vs odd indices). Save both adapters. Load the base model once, then swap adapters and verify that each produces different outputs on the same input. This is how production systems serve multiple fine-tuned models from one base.

5. **Merge vs. unmerged inference.** Compare the output of the LoRA model before and after merge_lora_weights on the same 100 inputs. Verify the outputs are identical (within floating-point tolerance of 1e-5). Then benchmark inference speed for both -- merged should be slightly faster since it's a single matrix multiply instead of two.